# Curve & fly screener — every two-leg curve and butterfly, spot and forward

Ranks 1,075 structures on static-curve carry-and-roll and a risk-adjusted version
of it. One leg warm composes into every structure, so the whole screen prices in
about a second.

## Read these four things before you read the ranking

1. **The carry convention is the whole design decision.** Under *forwards-realised*
   a par swap's carry-and-roll is identically zero — it is the arbitrage-free
   statement, not a signal. This uses the **static-curve** convention. The trap is
   the spot leg: ageing a `T`y swap to the `h x (T-h)` **forward** instead of the
   `(T-h)` **spot** silently reimplements forwards-realised. Gate G3 exists purely
   to catch that.
2. **`IRSwapValue.CARRY_AND_ROLL_BPS_RUNNING` is not this quantity** and is never
   used in the ranking. Against the published bank screen it correlates −0.136
   where a repriced roll scores +0.991. On `10y10y/20y10y` it disagrees by 10bp
   *and flips the sign*.
3. **Carry and richness are largely the same fact.** `corr(cr_bp, zs) = +0.61`.
   Carry-and-roll here *is* the aged level minus the level, so a structure at an
   extreme of its range shows fat roll for the same reason it looks rich. Ranking
   on `rac` alone systematically surfaces things that have already run — hence
   `rac_net`, and hence reading both.
4. **47 structures are degenerate.** Their legs sit between the same curve nodes
   and move in lockstep, so the level is near-constant by construction and the
   ratio explodes. Before the floor they were the *top and bottom* of the ranking.

Nothing is reported until the gates in section 4 pass.

In [1]:
import os

os.environ.setdefault("ARBS_SUPABASE_ENABLED", "0")

import dataclasses
import datetime
import math
import pathlib
import pickle
import sys
import time

import numpy as np
import pandas as pd

_here = pathlib.Path.cwd()
_REPO = next(p for p in [_here, *_here.parents] if (p / "RVUtils").is_dir())
sys.path.insert(0, str(_REPO))

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = "notebook_connected"
pd.set_option("display.width", 220)
pd.set_option("display.max_rows", 120)
print("repo:", _REPO)

repo: C:\Users\chris\clee\ARBS-fly


## 1. CONFIG — every knob, and why it is set where it is

In [2]:
@dataclasses.dataclass(frozen=True)
class ScreenConfig:
    """Everything this notebook chooses over and above the module defaults."""

    # ---- marks --------------------------------------------------------------
    as_of: datetime.datetime = datetime.datetime(2026, 8, 21, 17, 0)
    """Curve date. Naive is localised to America/New_York, which is the zone Citi
    Velocity's own stamps are in."""
    curve: str = "USD-SOFR-1D"
    source: str = "citivelo_excel_rl"

    # ---- history ------------------------------------------------------------
    hist_start: datetime.date = datetime.date(2024, 8, 21)
    hist_end: datetime.date = datetime.date(2026, 8, 21)
    """Two years. Long enough for a stable vol and z-score, short enough that the
    sample is the current regime. Warming ~70 legs over this window costs ~520s
    the first time and nothing after; the parquet is gitignored and regenerable."""

    # ---- measure ------------------------------------------------------------
    horizon_y: float = 1.0
    """Carry-and-roll horizon. 1y matches the ConvexityRV work so the numbers are
    directly comparable to that book."""
    business_days: float = 252.0

    # ---- gates --------------------------------------------------------------
    vol_floor_bp: float = 0.25
    """Realised daily vol below which a structure is treated as a curve-
    interpolation artifact rather than a trade. 47 structures fail this and they
    would otherwise occupy both ends of the ranking."""
    min_obs: int = 100
    """Below this, a z-score is computed off too few points to mean anything."""
    compose_tol_bp: float = 3.0
    """Compose-vs-price tolerance. The TimeseriesBuilder history and the live
    pricer are different vintages; a small gap is expected, a large one means the
    two sources have diverged and the z-scores cannot be trusted."""


CFG = ScreenConfig()
LEG_HIST = _REPO / "docs" / "curvefly" / "leg_history.parquet"
CFG

ScreenConfig(as_of=datetime.datetime(2026, 8, 21, 17, 0), curve='USD-SOFR-1D', source='citivelo_excel_rl', hist_start=datetime.date(2024, 8, 21), hist_end=datetime.date(2026, 8, 21), horizon_y=1.0, business_days=252.0, vol_floor_bp=0.25, min_obs=100, compose_tol_bp=3.0)

## 2. Leg warm

Every structure level is linear in its legs, so one warm over ~70 legs composes
into all 1,075 with no per-structure fetch. Goes through the repo pattern:
`IRSwapsMDP → IRSwapsTB → TimeseriesBuilder`, one `UnifiedQuery` per leg.

Tenor shorthand stays **lowercase-concatenated** (`10y10y`) throughout — tenor
case forks the cache symbol, so mixing `10Y10Y` in would silently double the
fetch and split the history.

In [3]:
from RVUtils.CurveFlyScreener import full_universe, leg_label, leg_universe

LEGS = leg_universe()
print(f"{len(LEGS)} legs")

if LEG_HIST.exists():
    leg_hist = pd.read_parquet(LEG_HIST)
    print(f"loaded cached warm: {leg_hist.shape}")
else:
    import pytz

    from MDP.IRSwaps.IRSwapsMDP import IRSwapsMDP
    from Query.Unified.registry import UnifiedValue
    from Query.Unified.UnifiedQuery import UnifiedQuery
    from TB.IRSwapsTB import IRSwapsTB
    from TB.TimeseriesBuilder import TimeseriesBuilder

    NY = pytz.timezone("America/New_York")
    labels = [leg_label(l) for l in LEGS]
    t0 = time.time()
    raw = TimeseriesBuilder().get_timeseries(
        start=NY.localize(datetime.datetime.combine(CFG.hist_start, datetime.time(17))),
        end=NY.localize(datetime.datetime.combine(CFG.hist_end, datetime.time(17))),
        queries=[UnifiedQuery(curve=CFG.curve, tenor=t, value=UnifiedValue.IRS_RATE)
                 for t in labels],
        n_jobs=8,
        routers={"IRS": IRSwapsTB(IRSwapsMDP(source=CFG.source), show_tqdm=False)},
    )
    ren = {f"{CFG.curve} {lab} OUTRIGHT RATE": lab for lab in labels}
    leg_hist = raw[[c for c in ren if c in raw.columns]].rename(columns=ren) * 100.0
    LEG_HIST.parent.mkdir(parents=True, exist_ok=True)
    leg_hist.to_parquet(LEG_HIST)
    print(f"warmed in {time.time()-t0:.0f}s -> {leg_hist.shape}")

cov = leg_hist.notna().mean()
print(f"{len(leg_hist)} dates | median leg coverage {cov.median():.0%}")
thin = cov[cov < 0.8]
if len(thin):
    print("THIN LEGS (excluded from vol/z by the min_obs gate):")
    print((thin * 100).round(0).to_string())
else:
    print("no thin legs")

71 legs
loaded cached warm: (500, 71)
500 dates | median leg coverage 100%
no thin legs


## 3. Universe and pricing

Bounds are stated rather than silently applied: forward legs capped at
`start + tenor <= 40` to stay on the liquid curve, and the general
`f1 x t1` vs `f2 x t2` cross product (~2,000 mostly unquoted names) is excluded
by design rather than allowed to dominate the ranking by sheer count.

Par rates are memoised by `(fwd, tenor)`: the screen touches ~70 distinct legs
today and ~70 aged, so without the memo the same handful of swaps get rebuilt and
repriced thousands of times.

In [4]:
from MDP.IRSwaps.IRSwapsMDP import IRSwapsMDP
from RVUtils.CurveFlyScreener import (
    add_risk_adjustment, compose_levels, screen, structure_rate_bp,
)

pricer = IRSwapsMDP(source=CFG.source).get_data(
    {"curve_name": CFG.curve, "timestamp": CFG.as_of})
UNI = full_universe()
print(pd.Series([s.kind for s in UNI]).value_counts().to_string())

t0 = time.time()
raw_screen = screen(pricer, UNI, horizon_y=CFG.horizon_y)
print(f"\npriced {len(raw_screen)} structures in {time.time()-t0:.1f}s")
raw_screen = raw_screen[raw_screen.error == ""].drop(
    columns=["error", "rlzd_vol_bp", "zs_1y", "rac"])
levels = compose_levels(leg_hist, UNI)
print(f"composed {levels.shape[1]} level histories over {len(levels)} dates")

C:\Users\chris\clee\ARBS-fly\MDP\CitiVelocityExcel\curves\rl_builder.py:79: LicenceNotice:


Rateslib is source-available (not open-source) software distributed under a dual-licence model.
No commercial licence is registered for this installation. Use is therefore permitted for non-commercial purposes only (at-home or university based academic use).
Any use in commercial, professional, or for-profit environments, including evaluation or trial use, requires a valid commercial licence or an approved evaluation licence.
Certain features may require a registered commercial or evaluation licence in current or future versions.
For licensing information or to register a licence, please visit: https://rateslib.com/licence



C:\Users\chris\clee\ARBS-fly\MDP\IRSwaps\CITIVELO_EXCEL\timestamps.py:261: UserWarning:

citivelo_excel: naive timestamp Timestamp('2026-08-21 17:00:00') was localised to America/New_York, which is the zone Citi Velocity's own stamps are in (measured 2026-08-07). Pass a tz-aware datetime to say so explicitly, or set CITIVELO_EXCEL_STRICT_TZ=1 to make this an error. Further naive timestamps this process will not be warned about.



fly_fwd            532
curve_fwd_start    236
curve_fwd_tenor    178
fly_spot            84
curve_spot          45


C:\Users\chris\anaconda3\envs\stir\Lib\site-packages\rateslib\data\fixings.py:3426: RuntimeWarning:

invalid value encountered in divide




priced 1075 structures in 1.1s
composed 1075 level histories over 500 dates


## 4. GATES

Two, and both have fired for real.

**compose-vs-price** — a level built from the leg warm against the same level
priced off the live curve. They come from different sources at slightly different
vintages, so a small gap is expected. A large one means the z-scores are being
computed against a history that no longer describes today's level.

**degeneracy** — legs read off the same curve nodes move in lockstep, so a
structure spanning no node has a near-constant level and a near-zero denominator.
Before this floor, `20y(5s7s10s)` scored `rac` 6.92 on 0.061bp/day and
`20y(2s3s)` scored −7.19 on 0.035. They were the best and worst names on the
board and neither exists.

In [5]:
# ---- GATE 1: compose vs price -------------------------------------------
chk = [(s.label, structure_rate_bp(pricer, s), float(levels[s.label].ffill().iloc[-1]))
       for s in UNI[::19] if s.label in levels.columns]
g1 = pd.DataFrame(chk, columns=["label", "priced", "composed"])
g1["diff"] = g1.priced - g1.composed
mean_d, max_d = g1["diff"].abs().mean(), g1["diff"].abs().max()
print(f"GATE compose-vs-price on {len(g1)}: mean |diff| {mean_d:.2f}bp | max {max_d:.2f}bp")
print(g1.reindex(g1["diff"].abs().sort_values(ascending=False).index)
      .head(4).round(2).to_string(index=False))
if max_d > CFG.compose_tol_bp:
    print(f"\n  !! above the {CFG.compose_tol_bp}bp tolerance. The history and the live")
    print("     pricer have diverged; z-scores are computed WITHIN the composed")
    print("     history so they stay internally consistent, but do not mix the two.")
else:
    print(f"  within the {CFG.compose_tol_bp}bp tolerance")

df = add_risk_adjustment(raw_screen, levels, business_days=CFG.business_days,
                         min_obs=CFG.min_obs)

# ---- GATE 2: degeneracy --------------------------------------------------
deg = df[df.rlzd_vol_bp < CFG.vol_floor_bp]
print(f"\nGATE degeneracy: {len(deg)} structures under {CFG.vol_floor_bp}bp/day, excluded")
if len(deg):
    print("  the rac they would have scored, worst first:")
    print(deg.reindex(deg.rac.abs().sort_values(ascending=False).index)
          .head(5)[["label", "level_bp", "cr_bp", "rlzd_vol_bp", "rac"]]
          .round(3).to_string(index=False))
df = df[df.rlzd_vol_bp >= CFG.vol_floor_bp].copy()
print(f"\n-> {len(df)} structures survive both gates")

GATE compose-vs-price on 57: mean |diff| 0.42bp | max 1.51bp
       label  priced  composed  diff
15y(2s3s20s)   52.02     50.51  1.51
10y(1s5s10s)   12.79     13.82 -1.02
5y(1s10s15s)   31.72     32.73 -1.01
   1y2y/5y2y   26.13     27.01 -0.88
  within the 3.0bp tolerance

GATE degeneracy: 47 structures under 0.25bp/day, excluded
  the rac they would have scored, worst first:
       label  level_bp   cr_bp  rlzd_vol_bp    rac
   20y(2s3s)    -0.051  -3.971        0.035 -7.194
20y(5s7s10s)    -2.696   6.713        0.061  6.922
 20y(1s3s5s)    -0.158 -12.354        0.113 -6.904
   20y(2s5s)    -0.124  -7.108        0.065 -6.859
   20y(1s3s)    -0.231 -15.491        0.143 -6.847

-> 1019 structures survive both gates


## 5. The confound

`corr(cr_bp, zs)` across the surviving universe. This is a property of the
measure, not of the sample: carry-and-roll *is* the aged level minus the level,
so a structure sitting at an extreme of its own range shows fat roll for the same
reason it looks rich.

The consequence is that `rac` alone is a ranking of *things that have already
moved*. `rac_net` charges the position for full reversion to its sample mean,
which is the conservative bound in the other direction. Read both.

In [6]:
ok = df.dropna(subset=["cr_bp", "zs"])
rho = float(np.corrcoef(ok.cr_bp, ok.zs)[0, 1])
print(f"corr(carry, level z) = {rho:+.3f} over {len(ok)} structures")
for k, sub in ok.groupby("kind"):
    if len(sub) > 5:
        print(f"  {k:<18} {float(np.corrcoef(sub.cr_bp, sub.zs)[0,1]):+.3f}  (n={len(sub)})")

fig = px.scatter(ok, x="zs", y="cr_bp", color="kind", hover_name="label",
                 labels={"zs": "level z-score (2y)", "cr_bp": "1y carry-and-roll, bp"},
                 title=f"Carry is not independent of entry level  (rho = {rho:+.2f})")
fig.add_hline(y=0, line_width=1, line_color="#888")
fig.add_vline(x=0, line_width=1, line_color="#888")
fig.update_layout(height=520, legend_title_text="")
fig.show()

corr(carry, level z) = +0.606 over 1019 structures
  curve_fwd_start    +0.568  (n=221)
  curve_fwd_tenor    +0.574  (n=177)
  curve_spot         +0.639  (n=36)
  fly_fwd            +0.486  (n=502)
  fly_spot           +0.717  (n=83)


## 6. The ranking

`rac` is the carry view. `rac_net` is the value view. They disagree, and the
disagreement is the output — not a defect to be averaged away.

In [7]:
NAMED = ["10y10y/20y10y", "10y10y/15y10y", "5y10y/10y10y", "15y5y/20y5y",
         "2s10s", "5s30s", "2s7s20s", "5s10s30s"]
COLS = ["label", "kind", "level_bp", "cr_bp", "rlzd_vol_bp", "zs", "rac",
        "rev_drag_bp", "cr_net_rev", "rac_net"]
print("=== the structures the desk names ===")
print(df[df.label.isin(NAMED)][COLS].round(2).to_string(index=False))

print("\n=== by family ===")
print(ok.groupby("kind").agg(
    n=("rac", "size"), mean_cr=("cr_bp", "mean"),
    pct_carry_pos=("cr_bp", lambda x: (x > 0).mean()),
    mean_rac=("rac", "mean"), mean_rac_net=("rac_net", "mean"),
    mean_vol=("rlzd_vol_bp", "mean")).round(3).to_string())

=== the structures the desk names ===
        label            kind  level_bp  cr_bp  rlzd_vol_bp    zs   rac  rev_drag_bp  cr_net_rev  rac_net
        2s10s      curve_spot     24.32   3.08         2.74  0.56  0.07       -10.95       -7.87    -0.18
        5s30s      curve_spot     41.05   3.06         2.85  0.30  0.07        -8.59       -5.53    -0.12
  15y5y/20y5y curve_fwd_tenor    -36.65   1.23         1.85  1.32  0.04        -7.62       -6.39    -0.22
 5y10y/10y10y curve_fwd_tenor     24.89   6.59         1.18  0.69  0.35        -7.29       -0.70    -0.04
10y10y/15y10y curve_fwd_tenor    -15.81   6.63         0.65  1.22  0.64        -7.75       -1.12    -0.11
10y10y/20y10y curve_fwd_tenor    -55.54   9.13         1.43  1.15  0.40       -10.79       -1.67    -0.07
      2s7s20s        fly_spot    -24.04   0.80         2.01  0.61  0.02        -6.33       -5.53    -0.17
     5s10s30s        fly_spot     -4.72  -5.80         0.93 -1.00 -0.39         7.40        1.60     0.11

=== by 

In [8]:
for kind in ok.kind.unique():
    sub = ok[ok.kind == kind]
    print(f"\n=== {kind} ===")
    for col in ("rac", "rac_net"):
        top = sub.sort_values(col, ascending=False).head(5)
        print(f"  top 5 by {col}:")
        print(top[["label", "level_bp", "cr_bp", "zs", "rac", "rac_net"]]
              .round(2).to_string(index=False))


=== curve_spot ===
  top 5 by rac:
 label  level_bp  cr_bp   zs  rac  rac_net
  2s3s      1.89   5.05 1.58 0.43    -0.22
  2s4s      3.67   5.16 1.28 0.26    -0.23
15s20s      8.40   1.98 0.46 0.26    -0.01
15s30s      5.61   3.46 0.62 0.24    -0.20
20s30s     -2.79   1.49 0.73 0.18    -0.34
  top 5 by rac_net:
 label  level_bp  cr_bp   zs  rac  rac_net
15s20s      8.40   1.98 0.46 0.26    -0.01
10s20s     25.67   2.94 0.32 0.16    -0.04
 7s20s     37.01   2.61 0.23 0.09    -0.05
10s15s     17.28   0.97 0.23 0.08    -0.06
 7s15s     28.62   0.64 0.16 0.03    -0.07

=== curve_fwd_start ===
  top 5 by rac:
     label  level_bp  cr_bp   zs  rac  rac_net
10y(2s30s)    -37.00  12.87 1.24 0.81    -0.16
10y(3s30s)    -40.71  12.02 1.24 0.78    -0.14
10y(1s30s)    -32.47  13.13 1.25 0.78    -0.19
10y(5s30s)    -45.54   9.98 1.27 0.72    -0.15
10y(2s20s)    -12.52  11.17 1.17 0.71    -0.05
  top 5 by rac_net:
     label  level_bp  cr_bp    zs  rac  rac_net
  1y(1s2s)     -0.69   7.62 -0.39 0.4

## 7. Carry against reversion, at the horizon

`rac_net` assumes reversion completes. That over-charges a slow structure, so
scale it by how much of the move actually lands inside the holding period, using
each level's own AR(1) half-life:

    fraction reverted over h with half-life L  =  1 - 2^(-h/L)

The half-lives are estimated in-sample and are the least stable number in this
notebook. Treat the direction as solid and the magnitudes as indicative.

In [9]:
def half_life_months(s: pd.Series, obs_per_month: float = 21.0) -> float:
    """AR(1) half-life of a level series, in months. inf when no mean reversion."""
    x = s.dropna()
    dx = x.diff().dropna()
    xl = (x.shift(1).dropna()).loc[dx.index]
    beta = float(np.polyfit(xl - x.mean(), dx, 1)[0])
    return (math.log(2) / -beta / obs_per_month) if beta < 0 else math.inf


HORIZONS = (3.0, 4.5, 6.0, 12.0)
rows = []
for lab in NAMED:
    if lab not in levels.columns or lab not in set(df.label):
        continue
    r = df[df.label == lab].iloc[0]
    L = half_life_months(levels[lab])
    row = {"structure": lab, "half_life_m": round(L, 1) if math.isfinite(L) else np.inf,
           "carry_1y": round(r.cr_bp, 2), "drag_full": round(r.rev_drag_bp, 2)}
    for h in HORIZONS:
        frac = 1 - 2 ** (-h / L) if math.isfinite(L) else 1.0
        row[f"net_{h:g}m"] = round(r.cr_bp * h / 12 + r.rev_drag_bp * frac, 2)
    rows.append(row)
hz = pd.DataFrame(rows)
print("bp of structure. Positive = a long position in the quoted level earns it.")
print(hz.to_string(index=False))

bp of structure. Positive = a long position in the quoted level earns it.
    structure  half_life_m  carry_1y  drag_full  net_3m  net_4.5m  net_6m  net_12m
10y10y/20y10y          3.1      9.13     -10.79   -3.00     -3.43   -3.41    -0.93
10y10y/15y10y          6.9      6.63      -7.75   -0.37     -0.35   -0.21     1.18
 5y10y/10y10y          4.5      6.59      -7.29   -1.03     -1.14   -1.07     0.47
  15y5y/20y5y          0.7      1.23      -7.62   -6.99     -7.09   -6.99    -6.39
        2s10s          2.0      3.08     -10.95   -6.29     -7.48   -8.03    -7.70
        5s30s          4.5      3.06      -8.59   -2.39     -3.12   -3.62    -4.16
      2s7s20s          1.6      0.80      -6.33   -4.45     -5.16   -5.48    -5.50
     5s10s30s          4.4     -5.80       7.40    1.35      1.60    1.65     0.50


## 8. What this does not know

* **Bid-offer.** The screen ranks carry per unit of realised vol and nothing else.
  The top raw-`rac` name is typically a 1y tail 10–20 years forward, which is
  nothing like as liquid as a 10y tail.
* **Whether the mean is the attractor.** `rac_net` assumes reversion to a two-year
  mean. If the curve is structurally repricing rather than mean-reverting, `rac`
  is the better guide. That is a view, not a screen output — but the trade
  requires one, and this notebook's job is to make that explicit rather than
  quietly pick a side.
* **Convexity.** Carry-and-roll is a first-order measure. A forward flattener is
  convex and a steepener concave; neither shows up here. For that, see the
  ConvexityRV work and the breakeven-vol statistic.
* Mean `rac_net` is ~0 or negative for **every** family. Charged for where levels
  sit, the whole screen is close to a wash. It sorts structures; it does not find
  free carry.